# Loading Libraries And Data


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os
import shutil

target_dir = '/content'
keep_file = 'arch_training_data.json'

for item in os.listdir(target_dir):
    item_path = os.path.join(target_dir, item)
    # Skip the file we want to keep
    if item == keep_file:
        continue
    try:
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
        else:
            os.remove(item_path)
    except Exception as e:
        print(f"Could not delete {item_path}: {e}")

In [ ]:

from unsloth import to_sharegpt, standardize_sharegpt
from datasets import load_dataset

# Correctly load the local JSON file howusing the 'data_files' parameter
dataset = load_dataset('json', data_files='/content/arch_training_data.json', split='train')

#Converting Data into ShareGpt

In [ ]:
dataset = to_sharegpt(
    dataset,
    merged_prompt="{title}\n{question}",
    output_column_name="answer"
)
dataset= standardize_sharegpt(dataset)

# Loading Model And Choosing the Matrices for QLora

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=4096,
    load_in_4bit=True,
    device_map="auto",
)

==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, #
    bias = "none",

    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


# Getting Qwen's Chat template and applying it to the whole dataset

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen2.5-coder",
)

adding the system prompt

In [ ]:
system_msg = "You are an expert Arch Linux systems engineer. Provide direct, concise, production-ready terminal solutions without conversational filler, tutorials, or forum signatures."
def inject_system_prompt(row):
    if row["conversations"][0]["role"] != "system":
        row["conversations"].insert(0, {"role": "system", "content": system_msg})
    return row
dataset = dataset.map(inject_system_prompt)

In [ ]:
import re

def clean_forum_noise(text):
    # 1. Truncate everything after a horizontal rule followed by forum markers
    # Combined (?i) and (?m) flags at the very start of the string to avoid PatternError
    text = re.sub(
        r"(?im)^[-*_]{3,}\s*\n+(edit|update|in response to|some additional reading|note:).*[\s\S]*$",
        "",
        text
    )

    # 2. Truncate standalone Edit/Update blocks anchored to the start of a line to the end of string
    text = re.sub(
        r"(?m)^(edit\s*\d*|update\s*\d*|in response to.*):\s*[\s\S]*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    # 3. Strip closing sign-offs and everything trailing them
    text = re.sub(
        r"(?i)\b(hope (this|my answer|that) helps|cheers|regards|best regards|thanks in advance)[\s\S]*$",
        "",
        text
    )

    # 4. Remove dangling dividers at the end of the text
    text = re.sub(r"(?m)^[-*_]{3,}\s*$", "", text)

    return text.strip()

def sanitize_dataset(row):
    for turn in row["conversations"]:
        if turn["role"] == "assistant":
            turn["content"] = clean_forum_noise(turn["content"])
    return row

dataset = dataset.map(sanitize_dataset)

In [ ]:
def format_chat(row):
    row["text"] = tokenizer.apply_chat_template(
        row["conversations"],
        tokenize=False,
        add_generation_prompt=False
    )
    return row

dataset = dataset.map(format_chat)

# Spliting the data and finetuning the model

In [ ]:
from sklearn.model_selection import train_test_split
split_dataset = dataset.train_test_split(test_size=0.05, shuffle=True, seed=3407)

train_data = split_dataset["train"] # The 95% the model learns from
eval_data = split_dataset["test"]   # The 5% used to test generalization

In [ ]:
train_data[0]

{'conversations': [{'content': 'You are an expert Arch Linux systems engineer. Provide direct, concise, production-ready terminal solutions without conversational filler, tutorials, or forum signatures.',
   'role': 'system'},
  {'content': 'I deleted my user directory, but I\'m not sure how\nI just installed a second hard drive. Previously my partition scheme was /dev/sda1 contained the boot files, and /dev/sda2 contained everything else. Now /dev/sda is the new hard drive, and /dev/sdb is the old hard drive. I want to use /dev/sda as my /home partition. So, I created the GPT partition table in /dev/sda with `gdisk` (single partition), then I formatted /dev/sda1 with `mkfs.ext4`, and then I added /dev/sda1 to my fstab as /home, ext4, "defaults", 0, and 2. I rebooted. I should have rebooted at this point, but I did something stupid and `mv`ed my only user folder, mszegedy, to /. I then `mount`ed /dev/sda1 in /home, and `mv`ed mszegedy back to /home. I then rebooted, to find that /home/

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        output_dir='/content',
        save_total_limit=1,       # keep only 3 most recent (disk saving)
        warmup_ratio = 0.1,
        num_train_epochs = 1, # Set this for 1 full training run.
        #max_steps = 60,
        learning_rate = 5e-5,
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
    ),
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

Unsloth: Auto-detected instruction_part = '<|im_start|>user\n' and response_part = '<|im_start|>assistant\n'
Unsloth: Removed 24 out of 1985 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


In [ ]:
im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")

# A) How many training examples still end with <|im_end|> in their labels?
ds = trainer.train_dataset
n_ok = sum(1 for ex in ds if im_end in ex["labels"][-3:])
print(f"{n_ok}/{len(ds)} examples train on <|im_end|>")
print("max_length =", trainer.args.max_length)

# B) What exactly is being trained on for one example?
ex = ds[0]
print(repr(tokenizer.decode([i for i, l in zip(ex["input_ids"], ex["labels"]) if l != -100])[-100:]))

1950/1961 examples train on <|im_end|>
max_length = 4096
"1, it made it completely invisible. I'll try to move it to /dev/sda1 some other way, then.<|im_end|>"


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,961 | Num Epochs = 1 | Total steps = 246
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,2.018400
40,2.037200
60,2.033000
80,1.881500
100,1.984900
120,2.012600
140,1.938500
160,1.935400
180,2.001600
200,1.848500


In [ ]:
trainer.save_model('/content')

In [ ]:
from unsloth import FastLanguageModel


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/checkpoint-246",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
# C) After training: does the model want to stop where the true answer ends?
import torch
sample = eval_data[1]  # pick one with a SHORT answer
text = tokenizer.apply_chat_template(sample["conversations"], tokenize=False)
prefix = text[: text.rindex("<|im_end|>")]
enc = tokenizer(prefix, return_tensors="pt").to("cuda")
with torch.no_grad():
    probs = model(**enc).logits[0, -1].float().softmax(-1)
top = probs.topk(5)
print("P(<|im_end|>) =", probs[im_end].item())
print([(tokenizer.decode(i), round(p.item(), 3)) for p, i in zip(top.values, top.indices)])

P(<|im_end|>) = 0.7938812375068665
[('<|im_end|>', 0.794), (' These', 0.015), (' But', 0.012), (' \n\n', 0.012), (' If', 0.01)]


In [ ]:
sample = eval_data[3]
user_prompt = [msg for msg in sample["conversations"] if msg["role"] == "user" or msg['role']=='system']

inputs = tokenizer.apply_chat_template(
    user_prompt,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict=True
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    use_cache = True,
    do_sample=True,
    temperature = 0.7,
    top_p=0.9,
)

generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

print("=== ACTUAL ANSWER ===")
print([msg["content"] for msg in sample["conversations"] if msg["role"] == "assistant"][0])
print("\n=== MODEL GENERATION ===")
print(generated_text)

=== ACTUAL ANSWER ===
Run `pacman -S archlinux-keyring` first, and then run `pacman -Syu`

=== MODEL GENERATION ===
<|im_start|>system
You are an expert Arch Linux systems engineer. Provide direct, concise, production-ready terminal solutions without conversational filler, tutorials, or forum signatures.<|im_end|>
<|im_start|>user
Error during update with pacman
running pacman -Syu today and this error occurred:


```

(627/627) checking package integrity                                                                                                           \[########################################################################################\] 100%

error: libspeechd: signature from "Alexander Epaneshnikov <[email@alex19ep.me](mailto:email@alex19ep.me)\>" is unknown trust

:: File /var/cache/pacman/pkg/libspeechd-0.11.1-2-x86\_64.pkg.tar.zst is corrupted (invalid or corrupted package (PGP signature)).

Do you want to delete it? \[Y/n\] n

error: mujs: signature from "Daniel Berm